# Desafio Final — ClearBank
## Análise de Transações Financeiras

Notebook desenvolvido para ler, validar, analisar e exportar o histórico mensal
de transações de clientes da fintech **ClearBank**.

---

### Instruções de uso

**1. Arquivo de entrada**

Antes de executar, o arquivo `transacoes.csv` deve estar **na mesma pasta deste notebook**,
com exatamente estas colunas (nesta ordem):

| Coluna | Tipo esperado | Descrição |
|---|---|---|
| `id` | inteiro | Identificador único da transação |
| `data` | texto | Data no formato `AAAA-MM-DD` |
| `cliente_id` | texto | Código do cliente (não pode ser vazio) |
| `tipo` | texto | `credito` ou `debito` |
| `valor` | decimal | Valor da transação (deve ser maior que 0) |
| `descricao` | texto | Descrição livre da operação |
| `categoria` | texto | Ex.: `salario`, `compra`, `transferencia` |

O arquivo deve conter, no mínimo:
- **15 registros válidos** distribuídos em **3 ou mais meses**;
- **5 registros inválidos**, para testar a validação;
- **2 transações acima de R$ 10.000,00**, para testar a detecção de suspeitas.

O arquivo deve ser salvo em **UTF-8** (por causa dos acentos em descrições como "Salário").

**2. Execução**

Execute as células **na ordem, de cima para baixo**. A célula do `main()` roda o
processo completo e gera o arquivo `relatorio.json` na mesma pasta.

**3. Saídas geradas**

- Relatório formatado no terminal (resumo da limpeza, métricas mensais e transações suspeitas);
- Arquivo `relatorio.json` com o resultado consolidado da análise.

## 1. Importações e constantes

In [ ]:
import csv
import json
import os
from datetime import datetime

# ----- Constantes de configuração -----
ARQUIVO_ENTRADA = "transacoes.csv"     # CSV exportado pela equipe de operações
ARQUIVO_SAIDA = "relatorio.json"       # JSON gerado pela análise

LIMITE_SUSPEITO = 10000.00             # Transações acima deste valor são sinalizadas

FORMATO_DATA = "%Y-%m-%d"              # Formato esperado da coluna "data" (AAAA-MM-DD)
TIPOS_VALIDOS = ("credito", "debito")  # Únicos valores aceitos na coluna "tipo"

print("Configuração carregada.")
print(f"Arquivo de entrada : {ARQUIVO_ENTRADA}")
print(f"Arquivo de saída   : {ARQUIVO_SAIDA}")
print(f"Limite suspeito    : R$ {LIMITE_SUSPEITO:.2f}")

### Conferência do arquivo de entrada

O erro mais comum ao rodar no Colab é esquecer de enviar o `transacoes.csv` — sem ele,
todas as células seguintes trabalham com zero linhas.

A célula abaixo confere se o arquivo está acessível. **No Colab**, se ele não estiver,
abre automaticamente o seletor de upload (envie o `transacoes.csv` e também o
`analise_pandas.py`, usado no requisito opcional 1).

In [ ]:
if os.path.exists(ARQUIVO_ENTRADA):
    print(f"OK: arquivo encontrado em {os.path.abspath(ARQUIVO_ENTRADA)}")

else:
    print(f"'{ARQUIVO_ENTRADA}' não está na pasta atual: {os.getcwd()}")
    try:
        # google.colab só existe dentro do Google Colab
        from google.colab import files

        print("Ambiente Colab detectado — selecione os arquivos para enviar.")
        files.upload()
        print("Upload concluído. Execute esta célula novamente para confirmar.")

    except ImportError:
        print("Ambiente local: copie o CSV para a pasta deste notebook")
        print("e execute esta célula novamente.")

## 2. Leitura do arquivo CSV

A função `ler_transacoes()` usa `csv.DictReader` para acessar as colunas **pelo nome**
e devolve a lista de linhas brutas (ainda sem validação).

**Tratamento de erro nº 1:** `try/except` para o caso de o arquivo não existir
(`FileNotFoundError`). Em vez de quebrar, a função avisa e devolve uma lista vazia.

In [ ]:
def ler_transacoes(caminho_arquivo):
    """Lê o CSV e retorna a lista de transações brutas (dicionários).

    Retorna uma lista vazia se o arquivo não existir ou não puder ser lido.
    """
    linhas_brutas = []

    try:
        with open(caminho_arquivo, mode="r", encoding="utf-8", newline="") as arquivo:
            leitor = csv.DictReader(arquivo)
            for linha in leitor:
                linhas_brutas.append(linha)

    except FileNotFoundError:
        print(f"[ERRO] Arquivo '{caminho_arquivo}' não encontrado.")
        print("       Verifique se ele está na mesma pasta deste notebook.")
        return []

    except UnicodeDecodeError:
        print(f"[ERRO] Não foi possível ler '{caminho_arquivo}' como UTF-8.")
        print("       Salve o arquivo novamente com codificação UTF-8.")
        return []

    return linhas_brutas


print("Função ler_transacoes() definida.")

In [ ]:
# Teste do tratamento de erro: arquivo que não existe (não deve quebrar o notebook)
_ = ler_transacoes("arquivo_que_nao_existe.csv")

# Leitura do arquivo real
linhas_brutas = ler_transacoes(ARQUIVO_ENTRADA)
print()
print(f"Linhas lidas do CSV: {len(linhas_brutas)}")

## 3. Validação e limpeza dos dados

A função `validar_transacao()` recebe **uma única linha** do CSV e devolve:

- um **dicionário limpo** (com `valor` já convertido para `float` e `data` para `datetime`), se a linha for válida;
- `None`, se a linha for inválida.

Regras de descarte (todas silenciosas — não interrompem o programa):

| Campo | Motivo de descarte |
|---|---|
| `id` | vazio ou não numérico |
| `cliente_id` | vazio |
| `data` | formato diferente de `AAAA-MM-DD` (ou data inexistente, ex.: `2026-06-31`) |
| `tipo` | diferente de `credito` ou `debito` |
| `valor` | não numérico ou menor/igual a zero |

**Tratamento de erro nº 2:** `try/except ValueError` na conversão de `valor` para `float`.

**Tratamento de erro nº 3:** `try/except ValueError` na conversão de `data` para `datetime`.

In [ ]:
def validar_transacao(linha):
    """Valida uma única linha do CSV.

    Retorna o registro limpo (dict) se for válida, ou None se for inválida.
    """
    # A linha pode vir com colunas faltando; .get() evita KeyError
    id_bruto = (linha.get("id") or "").strip()
    data_bruta = (linha.get("data") or "").strip()
    cliente_id = (linha.get("cliente_id") or "").strip()
    tipo = (linha.get("tipo") or "").strip().lower()
    valor_bruto = (linha.get("valor") or "").strip()
    descricao = (linha.get("descricao") or "").strip()
    categoria = (linha.get("categoria") or "").strip().lower()

    # --- id: não pode ser vazio nem não numérico ---
    try:
        id_transacao = int(id_bruto)
    except ValueError:
        return None

    # --- cliente_id: não pode ser vazio ---
    if not cliente_id:
        return None

    # --- tipo: só credito ou debito ---
    if tipo not in TIPOS_VALIDOS:
        return None

    # --- data: precisa estar em AAAA-MM-DD e existir no calendário ---
    try:
        data = datetime.strptime(data_bruta, FORMATO_DATA)
    except ValueError:
        return None

    # --- valor: precisa ser numérico e maior que zero ---
    try:
        valor = float(valor_bruto.replace(",", "."))
    except ValueError:
        return None

    if valor <= 0:
        return None

    # Linha aprovada: devolve o registro já convertido e normalizado
    return {
        "id": id_transacao,
        "data": data,                       # objeto datetime
        "mes": data.strftime("%Y-%m"),      # chave de agrupamento mensal
        "cliente_id": cliente_id,
        "tipo": tipo,
        "valor": valor,                     # float
        "descricao": descricao,
        "categoria": categoria,
    }


# ----- Teste rápido: uma linha válida e uma inválida -----
linha_ok = {
    "id": "1", "data": "2026-01-05", "cliente_id": "CLI001", "tipo": "credito",
    "valor": "3500.00", "descricao": "Salário janeiro", "categoria": "salario",
}
linha_ruim = {
    "id": "3", "data": "2026-01-20", "cliente_id": "CLI001", "tipo": "debito",
    "valor": "abc", "descricao": "Erro de sistema", "categoria": "compra",
}

print("Linha válida   ->", validar_transacao(linha_ok))
print("Linha inválida ->", validar_transacao(linha_ruim))

In [ ]:
def limpar_transacoes(linhas_brutas):
    """Aplica validar_transacao() em todas as linhas e remove duplicatas.

    Duplicata = mesma data, cliente, tipo, valor, descrição e categoria.
    O campo 'id' é ignorado na comparação, porque registros repetidos pelo
    sistema de origem chegam com ids diferentes.

    Retorna: (lista_de_validas, qtd_invalidas, qtd_duplicadas)
    """
    validas = []
    chaves_vistas = set()
    qtd_invalidas = 0
    qtd_duplicadas = 0

    for linha in linhas_brutas:
        registro = validar_transacao(linha)

        if registro is None:
            qtd_invalidas += 1
            continue

        chave = (
            registro["data"],
            registro["cliente_id"],
            registro["tipo"],
            registro["valor"],
            registro["descricao"],
            registro["categoria"],
        )

        if chave in chaves_vistas:
            qtd_duplicadas += 1
            continue

        chaves_vistas.add(chave)
        validas.append(registro)

    return validas, qtd_invalidas, qtd_duplicadas


# ----- Teste rápido: 3 linhas, sendo 1 inválida e 1 duplicada -----
amostra = [linha_ok, linha_ruim, dict(linha_ok)]
teste_validas, teste_invalidas, teste_duplicadas = limpar_transacoes(amostra)
print(f"Amostra de {len(amostra)} linhas -> válidas: {len(teste_validas)}, "
      f"inválidas: {teste_invalidas}, duplicadas: {teste_duplicadas}")

In [ ]:
def exibir_resumo_limpeza(total_lidas, qtd_validas, qtd_invalidas, qtd_duplicadas):
    """Imprime no terminal o resumo da etapa de limpeza."""
    print("=" * 45)
    print("RESUMO DA LIMPEZA")
    print("=" * 45)
    print(f"Total de linhas lidas: {total_lidas}")
    print(f"Linhas válidas: {qtd_validas}")
    print(f"Linhas inválidas: {qtd_invalidas}")
    print(f"Linhas duplicadas removidas: {qtd_duplicadas}")


# ----- Teste rápido com os números da amostra acima -----
exibir_resumo_limpeza(len(amostra), len(teste_validas), teste_invalidas, teste_duplicadas)

In [ ]:
# Executa a limpeza sobre as linhas lidas
transacoes_validas, total_invalidas, total_duplicadas = limpar_transacoes(linhas_brutas)

exibir_resumo_limpeza(
    total_lidas=len(linhas_brutas),
    qtd_validas=len(transacoes_validas),
    qtd_invalidas=total_invalidas,
    qtd_duplicadas=total_duplicadas,
)

## 4. Formatação de valores monetários

Função auxiliar que converte um `float` para o padrão brasileiro: `R$ 3.500,00`
(ponto como separador de milhar, vírgula como separador decimal).

In [ ]:
def formatar_moeda(valor):
    """Formata um número no padrão monetário brasileiro: R$ 1.234,56"""
    # f"{valor:,.2f}" gera 1,234.56 (padrão americano).
    # Trocamos os separadores usando "@" como marcador temporário.
    texto = f"{valor:,.2f}"
    texto = texto.replace(",", "@").replace(".", ",").replace("@", ".")
    return f"R$ {texto}"


# Teste rápido
for exemplo in (3500.0, 180.5, 15000.0, 1840.25):
    print(formatar_moeda(exemplo))

## 5. Geração das métricas

A função `gerar_relatorio()` percorre as transações válidas e monta:

- **Período analisado**: data mais antiga, data mais recente e quantos **dias** se passaram entre elas;
- **Resumo mensal** (agrupado por `AAAA-MM`): quantidade, total de crédito, total de débito,
  saldo (crédito − débito), valor médio, maior e menor valor;
- **Transações suspeitas**: toda transação com `valor` acima de `LIMITE_SUSPEITO`.

In [ ]:
def gerar_relatorio(transacoes, qtd_invalidas):
    """Agrupa as transações por mês e calcula todas as métricas do desafio."""

    # --- Agrupamento mensal ---
    # Estrutura auxiliar: {"2026-01": [transacao, transacao, ...], ...}
    agrupado_por_mes = {}
    for transacao in transacoes:
        mes = transacao["mes"]
        if mes not in agrupado_por_mes:
            agrupado_por_mes[mes] = []
        agrupado_por_mes[mes].append(transacao)

    # --- Métricas de cada mês ---
    resumo_mensal = {}
    for mes in sorted(agrupado_por_mes):
        do_mes = agrupado_por_mes[mes]

        total_credito = 0.0
        total_debito = 0.0
        for t in do_mes:
            if t["tipo"] == "credito":
                total_credito += t["valor"]
            else:
                total_debito += t["valor"]

        valores = [t["valor"] for t in do_mes]
        quantidade = len(do_mes)

        resumo_mensal[mes] = {
            "quantidade": quantidade,
            "total_credito": round(total_credito, 2),
            "total_debito": round(total_debito, 2),
            "saldo": round(total_credito - total_debito, 2),
            "media": round(sum(valores) / quantidade, 2),
            "maior_valor": round(max(valores), 2),
            "menor_valor": round(min(valores), 2),
        }

    # --- Transações suspeitas ---
    suspeitas = []
    for t in transacoes:
        if t["valor"] > LIMITE_SUSPEITO:
            suspeitas.append({
                "id": t["id"],
                "cliente_id": t["cliente_id"],
                "data": t["data"].strftime(FORMATO_DATA),
                "valor": round(t["valor"], 2),
            })
    suspeitas.sort(key=lambda s: s["data"])

    # --- Período analisado (usando datetime) ---
    if transacoes:
        datas = [t["data"] for t in transacoes]
        data_inicial = min(datas)
        data_final = max(datas)
        periodo = {
            "data_inicial": data_inicial.strftime(FORMATO_DATA),
            "data_final": data_final.strftime(FORMATO_DATA),
            "dias_analisados": (data_final - data_inicial).days,
        }
    else:
        periodo = {"data_inicial": None, "data_final": None, "dias_analisados": 0}

    return {
        "gerado_em": datetime.now().strftime(FORMATO_DATA),
        "total_transacoes_validas": len(transacoes),
        "total_transacoes_invalidas": qtd_invalidas,
        "periodo": periodo,
        "limite_suspeito": LIMITE_SUSPEITO,
        "resumo_mensal": resumo_mensal,
        "transacoes_suspeitas": suspeitas,
    }


# ----- Teste rápido: gera o relatório e mostra as chaves e o 1º mês -----
relatorio_teste = gerar_relatorio(transacoes_validas, total_invalidas)
print("Chaves do relatório:", list(relatorio_teste.keys()))
print("Meses encontrados:", list(relatorio_teste["resumo_mensal"].keys()))

# O relatório pode vir vazio se o CSV não tiver sido lido; a célula não pode quebrar
if relatorio_teste["resumo_mensal"]:
    primeiro_mes = list(relatorio_teste["resumo_mensal"])[0]
    print(f"Métricas de {primeiro_mes}:", relatorio_teste["resumo_mensal"][primeiro_mes])
else:
    print("[AVISO] Nenhum mês para exibir: o arquivo 'transacoes.csv' não foi lido.")
    print("        Coloque o CSV na mesma pasta e execute as células desde o início.")

## 6. Exibição formatada no terminal

Separadores visuais entre as seções, valores monetários com `R$` e duas casas decimais,
período analisado e totais de transações válidas/inválidas.

In [ ]:
def exibir_relatorio(relatorio):
    """Imprime o relatório completo, formatado, no terminal."""

    # ---------- Cabeçalho ----------
    print()
    print("=" * 45)
    print("      CLEARBANK - ANÁLISE DE TRANSAÇÕES")
    print("=" * 45)

    periodo = relatorio["periodo"]
    if periodo["data_inicial"]:
        print(f"Período analisado: {periodo['data_inicial']} -> {periodo['data_final']}")
        print(f"Dias entre a mais antiga e a mais recente: {periodo['dias_analisados']}")
    else:
        print("Período analisado: nenhuma transação válida.")

    print(f"Transações válidas:   {relatorio['total_transacoes_validas']}")
    print(f"Transações inválidas: {relatorio['total_transacoes_invalidas']}")
    print(f"Relatório gerado em:  {relatorio['gerado_em']}")

    # ---------- Relatório mensal ----------
    print()
    print("===== RELATÓRIO MENSAL =====")

    if not relatorio["resumo_mensal"]:
        print("Nenhuma transação válida para analisar.")
    else:
        for mes, dados in relatorio["resumo_mensal"].items():
            print(f"Mês: {mes}")
            print(f"  Transações: {dados['quantidade']}")
            print(f"  Total crédito: {formatar_moeda(dados['total_credito'])}")
            print(f"  Total débito:  {formatar_moeda(dados['total_debito'])}")
            print(f"  Saldo:         {formatar_moeda(dados['saldo'])}")
            print(f"  Média:         {formatar_moeda(dados['media'])}")
            print(f"  Maior valor:   {formatar_moeda(dados['maior_valor'])}")
            print(f"  Menor valor:   {formatar_moeda(dados['menor_valor'])}")
            print()

    # ---------- Transações suspeitas ----------
    print("===== TRANSAÇÕES SUSPEITAS =====")
    print(f"(acima de {formatar_moeda(relatorio['limite_suspeito'])})")

    suspeitas = relatorio["transacoes_suspeitas"]
    if not suspeitas:
        print("Nenhuma transação suspeita encontrada.")
    else:
        for s in suspeitas:
            print(
                f"ID: {s['id']} | Cliente: {s['cliente_id']} | "
                f"Data: {s['data']} | Valor: {formatar_moeda(s['valor'])}"
            )

    print()
    print("=" * 45)


# A função é chamada dentro de main(), na seção 8, para não repetir o
# relatório inteiro duas vezes na saída do notebook.
print("Função exibir_relatorio() definida.")

## 7. Exportação do relatório em JSON

Salva o dicionário do relatório em `relatorio.json`, com acentuação preservada
(`ensure_ascii=False`) e indentação legível.

In [ ]:
def salvar_json(relatorio, caminho_arquivo):
    """Salva o relatório no arquivo JSON indicado. Retorna True em caso de sucesso."""
    try:
        with open(caminho_arquivo, mode="w", encoding="utf-8") as arquivo:
            json.dump(relatorio, arquivo, ensure_ascii=False, indent=2)

    except OSError as erro:
        print(f"[ERRO] Não foi possível salvar '{caminho_arquivo}': {erro}")
        return False

    print(f"Relatório salvo em: {os.path.abspath(caminho_arquivo)}")
    return True


# ----- Teste rápido: salva em um arquivo temporário e confere o tamanho -----
if salvar_json(relatorio_teste, "_teste_relatorio.json"):
    print(f"Tamanho do arquivo de teste: {os.path.getsize('_teste_relatorio.json')} bytes")
    os.remove("_teste_relatorio.json")
    print("Arquivo de teste removido.")

## 8. Execução completa

A função `main()` orquestra todas as etapas na ordem:
leitura → validação/limpeza → métricas → exibição → exportação JSON.

In [ ]:
def main():
    """Executa o fluxo completo da análise."""

    # 1) Leitura
    brutas = ler_transacoes(ARQUIVO_ENTRADA)
    if not brutas:
        print("Processamento encerrado: nenhuma linha foi lida.")
        return None

    # 2) Validação e limpeza
    validas, invalidas, duplicadas = limpar_transacoes(brutas)
    exibir_resumo_limpeza(len(brutas), len(validas), invalidas, duplicadas)

    # 3) Métricas
    relatorio = gerar_relatorio(validas, invalidas)

    # 4) Exibição no terminal
    exibir_relatorio(relatorio)

    # 5) Exportação
    salvar_json(relatorio, ARQUIVO_SAIDA)

    return relatorio


relatorio_final = main()

## 9. Conferência do JSON gerado

Célula opcional: relê o arquivo salvo em disco para confirmar que ele foi
gravado corretamente e mostra o início do conteúdo.

In [ ]:
try:
    with open(ARQUIVO_SAIDA, mode="r", encoding="utf-8") as arquivo:
        conteudo = json.load(arquivo)

    print(f"Arquivo '{ARQUIVO_SAIDA}' lido com sucesso.")
    print(f"Meses no resumo: {list(conteudo['resumo_mensal'].keys())}")
    print(f"Suspeitas registradas: {len(conteudo['transacoes_suspeitas'])}")
    print()
    print("--- Prévia do JSON ---")
    print(json.dumps(conteudo, ensure_ascii=False, indent=2)[:900] + "\n...")

except FileNotFoundError:
    print(f"[ERRO] '{ARQUIVO_SAIDA}' não foi encontrado. Execute a célula do main() antes.")

---

# Requisitos Opcionais

## 10. RO1 — Análise alternativa com pandas

A versão em pandas está no arquivo separado **`analise_pandas.py`**, para não
misturar com a solução principal (que usa apenas módulos nativos).

Esse arquivo:
1. carrega os dados com `pd.read_csv()`;
2. aplica **as mesmas regras de validação**, de forma vetorizada;
3. agrupa por mês com `groupby()` e calcula as sete métricas;
4. **compara** métrica a métrica com o `resumo_mensal` da solução nativa.

> **No Google Colab:** envie também o `analise_pandas.py` (menu lateral → Arquivos → Upload),
> senão o `import` abaixo falha. Se o pandas não estiver instalado localmente:
> `pip install pandas matplotlib`

In [ ]:
def executar_comparacao_pandas():
    """Roda a análise em pandas e compara com o resultado da solução nativa."""
    if relatorio_final is None:
        print("[AVISO] O relatório nativo não foi gerado; não há com o que comparar.")
        print("        Coloque 'transacoes.csv' na mesma pasta e rode a célula do main().")
        return

    try:
        from analise_pandas import analisar_com_pandas, comparar_resultados
    except ImportError as erro:
        print(f"[AVISO] Não foi possível importar a análise com pandas: {erro}")
        print("        Verifique se 'analise_pandas.py' está na mesma pasta")
        print("        e se o pandas está instalado (pip install pandas).")
        return

    resumo_pandas = analisar_com_pandas(ARQUIVO_ENTRADA)
    comparar_resultados(resumo_pandas, relatorio_final["resumo_mensal"])


executar_comparacao_pandas()

## 11. RO2 — Visualização com matplotlib

Gera o arquivo **`grafico.png`** com dois painéis:

- **Painel superior (Opção C):** barras empilhadas de crédito e débito por mês;
- **Painel inferior (Opção A):** saldo mensal (crédito − débito), em verde quando
  positivo e vermelho quando negativo.

Ambos com título, rótulos nos eixos e grade; o painel superior tem legenda.

In [ ]:
def gerar_grafico():
    """Monta o grafico.png a partir do resumo mensal do relatório."""
    if relatorio_final is None or not relatorio_final["resumo_mensal"]:
        print("[AVISO] Sem dados para plotar: o resumo mensal está vazio.")
        print("        Coloque 'transacoes.csv' na mesma pasta e rode a célula do main().")
        return

    try:
        import matplotlib.pyplot as plt
    except ImportError as erro:
        print(f"[AVISO] matplotlib não disponível: {erro}")
        print("        Instale com: pip install matplotlib")
        return

    resumo = relatorio_final["resumo_mensal"]
    meses = list(resumo.keys())
    creditos = [resumo[m]["total_credito"] for m in meses]
    debitos = [resumo[m]["total_debito"] for m in meses]
    saldos = [resumo[m]["saldo"] for m in meses]

    VERDE = "#2e7d32"
    VERMELHO = "#c62828"

    figura, (painel_topo, painel_base) = plt.subplots(2, 1, figsize=(10, 8))

    # ---------- Painel superior: barras empilhadas ----------
    painel_topo.bar(meses, creditos, label="Crédito", color=VERDE)
    painel_topo.bar(meses, debitos, bottom=creditos, label="Débito", color=VERMELHO)
    painel_topo.set_title("ClearBank — Crédito e débito por mês", fontsize=13, weight="bold")
    painel_topo.set_xlabel("Mês")
    painel_topo.set_ylabel("Valor (R$)")
    painel_topo.legend()
    painel_topo.grid(axis="y", linestyle="--", alpha=0.4)
    # Folga no topo para a barra mais alta não encostar na borda
    painel_topo.set_ylim(0, max(c + d for c, d in zip(creditos, debitos)) * 1.12)

    # ---------- Painel inferior: saldo mensal ----------
    cores_saldo = [VERDE if saldo >= 0 else VERMELHO for saldo in saldos]
    painel_base.bar(meses, saldos, color=cores_saldo)
    painel_base.axhline(0, color="black", linewidth=0.8)
    painel_base.set_title("Saldo mensal (crédito - débito)", fontsize=13, weight="bold")
    painel_base.set_xlabel("Mês")
    painel_base.set_ylabel("Saldo (R$)")
    painel_base.grid(axis="y", linestyle="--", alpha=0.4)
    # Folga no topo e embaixo para os rótulos de valor caberem
    limite = max(abs(s) for s in saldos) * 1.18
    painel_base.set_ylim(min(0, min(saldos) - limite * 0.1), limite)

    # Rótulo do valor em cima de cada barra de saldo
    for indice, saldo in enumerate(saldos):
        painel_base.text(
            indice, saldo, formatar_moeda(saldo),
            ha="center", va="bottom" if saldo >= 0 else "top", fontsize=8,
        )

    figura.tight_layout()
    figura.savefig("grafico.png", dpi=150, bbox_inches="tight")
    print("Gráfico salvo em: grafico.png")
    plt.show()


gerar_grafico()